In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Raghav\\Desktop\\Image_Captioning_using_DeepLearning\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Raghav\\Desktop\\Image_Captioning_using_DeepLearning'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    images_dir: Path
    captions_file: Path
    tokenizer_path: Path
    train_img_id_path: Path
    test_img_id_path: Path
    train_imagesid_captions_path: Path
    test_imagesid_captions_path: Path

    SPLIT_SIZE: float
    RANDOM_STATE: int


In [6]:
from imgCaption.constants import *
from imgCaption.utils.common import read_yaml,create_directories

In [7]:
class ConfigurationManager:
    def __init__(self,config_filepath=CONFIG_FILE_PATH,params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:

        config = self.config.data_transformation
        params = self.params

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
                                                    root_dir=Path(config.root_dir),
                                                    images_dir=Path(config.images_dir),
                                                    captions_file=Path(config.captions_file),
                                                    tokenizer_path=Path(config.tokenizer_path),
                                                    train_img_id_path=Path(config.train_img_id_path),                                            
                                                    test_img_id_path=Path(config.test_img_id_path),
                                                    train_imagesid_captions_path=Path(config.train_imagesid_captions_path),
                                                    test_imagesid_captions_path=Path(config.test_imagesid_captions_path),
                                                    SPLIT_SIZE=params.SPLIT_SIZE,
                                                    RANDOM_STATE=params.RANDOM_STATE
                                                )

        return data_transformation_config

In [8]:
import os
import re
import json
import pickle
import yaml
from imgCaption import logger
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer

In [9]:
class DataTransformation:
    def __init__(self,config : DataTransformationConfig):
        self.config = config


    def cleaned_captions(self):
        with open(self.config.captions_file) as f:
            raw_caption = f.readlines()

        captions = [cap.lower().split(',')[1] for cap in raw_caption[1:]]

        cleaned_captions = [
        re.sub(r'\s+', ' ',          
        re.sub(r'\d+', '',            
        re.sub(r'[^\w\s]', '', cap)   
        )).strip().lower()            
        for cap in captions
        ]

        images_ID = [ID.split(',')[0] for ID in raw_caption[1:]]

        full_cleaned_captions = []
        all_captions = []

        for img_id, caption in zip(images_ID, cleaned_captions):
            formatted_str = f"{img_id}\tstartseq {caption} endseq"
            cap_str = f"startseq {caption} endseq"
            full_cleaned_captions.append(formatted_str)
            all_captions.append(cap_str)
    
    
        logger.info("captions cleaned and start & end tokens added successfully")

        return full_cleaned_captions,all_captions


    def build_tokenizer(self, all_captions : list):
        self.tokenizer = Tokenizer()
        self.tokenizer.fit_on_texts(all_captions)

        with open(self.config.tokenizer_path, 'wb') as f:
            pickle.dump(self.tokenizer, f)

        logger.info(f"Tokenizer saved at: {self.config.tokenizer_path}")

        vocab_size = len(self.tokenizer.word_index) + 1    
        max_length = max(len(cap.split()) for cap in all_captions)

        logger.info(f"VOCAB_SIZE: {vocab_size}")
        logger.info(f"MAX_LENGTH: {max_length}")
        params_path = "params.yaml"
    
        try:
            with open(params_path, 'r') as file:
                existing_params = yaml.safe_load(file) or {}
        except FileNotFoundError:
            existing_params = {}
                    
        existing_params['VOCAB_SIZE'] = vocab_size
        existing_params['MAX_LENGTH'] = max_length
        
        with open(params_path, 'w') as file:
            yaml.safe_dump(existing_params, file, default_flow_style=None, sort_keys=False)
            
        logger.info(f"Successfully updated VOCAB_SIZE and MAX_LENGTH in {params_path}")
        
    def split_images_ID(self):
        images_Id = os.listdir(self.config.images_dir)
        train_image_Id, test_image_Id = train_test_split(images_Id, test_size=self.config.SPLIT_SIZE, random_state=self.config.RANDOM_STATE)
        
        logger.info(f"Total images     : {len(images_Id)}")
        logger.info(f"Train images     : {len(train_image_Id)}")
        logger.info(f"Test images      : {len(test_image_Id)}")

        os.makedirs(self.config.root_dir, exist_ok=True)

        with open(self.config.train_img_id_path,"w") as f:
            for img_id in train_image_Id:
                f.write(f"{img_id}\n")

        logger.info("train_img_id_file created")

        with open(self.config.test_img_id_path,"w") as f:
            for img_id in test_image_Id:
                f.write(f"{img_id}\n")

        logger.info("test_img_id_file created")

    
    def create_mapping_dict(self, cleaned_captions: list, img_ids_path: Path, data_path: Path):
       
        with open(img_ids_path, "r") as f:
            target_ids = set(line.strip() for line in f.readlines())

        caption_map = {}

        for entry in cleaned_captions:

            img_id,caption = entry.split('\t')

            if img_id in target_ids:
                if img_id not in caption_map:
                    caption_map[img_id] = []

                caption_map[img_id].append(caption)

        logger.info(f"Successfully made {Path(img_ids_path).name} a dictionary")

        with open(data_path, 'w') as f:
            json.dump(caption_map, f, indent=2)

        logger.info(f"File : {Path(data_path).name} has saved successfully")


In [10]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    cleaned_captions,all_captions = data_transformation.cleaned_captions()
    data_transformation.build_tokenizer(all_captions)
    data_transformation.split_images_ID()      
    data_transformation.create_mapping_dict(cleaned_captions=cleaned_captions,
                                    img_ids_path=data_transformation_config.train_img_id_path,
                                    data_path=data_transformation_config.train_imagesid_captions_path)
except Exception as e:
    raise e

[2026-07-01 16:49:42,912: INFO: common: yaml file: config\config.yaml loaded successfully]


[2026-07-01 16:49:42,926: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-01 16:49:42,927: INFO: common: created directory at: artifacts]
[2026-07-01 16:49:42,932: INFO: common: created directory at: artifacts/data_transformation]
[2026-07-01 16:49:43,800: INFO: 3037923417: captions cleaned and start & end tokens added successfully]
[2026-07-01 16:49:45,290: INFO: 3037923417: Tokenizer saved at: artifacts\data_transformation\model_tokenizer.pkl]
[2026-07-01 16:49:45,369: INFO: 3037923417: VOCAB_SIZE: 8588]
[2026-07-01 16:49:45,369: INFO: 3037923417: MAX_LENGTH: 35]
[2026-07-01 16:49:45,376: INFO: 3037923417: Successfully updated VOCAB_SIZE and MAX_LENGTH in params.yaml]
[2026-07-01 16:49:45,408: INFO: 3037923417: Total images     : 8091]
[2026-07-01 16:49:45,408: INFO: 3037923417: Train images     : 6472]
[2026-07-01 16:49:45,410: INFO: 3037923417: Test images      : 1619]
[2026-07-01 16:49:45,423: INFO: 3037923417: train_img_id_file created]
[2026-07-01 16:49:45,441

In [1]:
import tensorflow as tf
print(tf.__version__)

2.12.0
